# Kasus 02: Konsumsi Energi Gedung
Jalankan dari folder `notebooks/`. Urutan: EDA, pembersihan, fitur, baseline, model, test, simpan.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error as mae
from features import bersihkan, buat_fitur, TARGET
raw = pd.read_csv('../data/raw.csv')
raw.info(); raw.describe().T

## EDA awal
Dua hal langsung terlihat: `konsumsi_kWh` punya nilai 99999 dan negatif, dan kolom `waktu` memakai dua format tanggal.

In [ ]:
print(raw.duplicated().sum(), 'baris duplikat')
print(raw.waktu.str.contains('/').sum(), 'baris berformat dd/mm/yyyy')
print((raw[TARGET] >= 9999).sum(), 'kode 99999;', (raw[TARGET] < 0).sum(), 'negatif;', (raw[TARGET] == 0).sum(), 'nol')
raw.isna().sum()

## Pembersihan
Semua aturan ada di `src/features.py` supaya sama persis dengan yang dipakai saat prediksi.

In [ ]:
d, log = bersihkan(raw)
log

In [ ]:
prof = d.groupby([d.index.hour, d.index.dayofweek >= 5])[TARGET].mean().unstack()
prof.columns = ['hari kerja', 'akhir pekan']; prof.plot(ylabel='rata-rata kWh', xlabel='jam');

Pola harian jelas: beban dasar sekitar 85 kWh di malam hari, naik saat jam kantor, dan akhir pekan hampir datar. Okupansi diperkirakan jadi fitur terkuat.

In [ ]:
X = buat_fitur(d); y = d[TARGET]; ok = y.notna()
tr = (X.index < '2026-03-01') & ok
va = (X.index >= '2026-03-01') & (X.index < '2026-05-01') & ok
te = (X.index >= '2026-05-01') & ok
print(tr.sum(), va.sum(), te.sum())

## Baseline dan model (diukur di validation)

In [ ]:
base = X['lag_168']; m = va & base.notna()
print('baseline lag 168:', mae(y[m], base[m]))
Xi = X.fillna(X.median())
print('ridge:', mae(y[va], Ridge(1.0).fit(Xi[tr], y[tr]).predict(Xi[va])))
kandidat = [dict(max_iter=200, learning_rate=.05, max_depth=4), dict(max_iter=400, learning_rate=.05, max_depth=6), dict(max_iter=300, learning_rate=.1, max_leaf_nodes=15)]
skor = []
for p in kandidat:
    g = HistGradientBoostingRegressor(loss='absolute_error', random_state=0, **p).fit(X[tr], y[tr])
    skor.append(mae(y[va], g.predict(X[va]))); print(p, skor[-1])
bp = kandidat[int(np.argmin(skor))]

## Test set: dibuka satu kali
Model dilatih ulang pada train + validation dengan parameter terbaik.

In [ ]:
trv = tr | va
final = HistGradientBoostingRegressor(loss='absolute_error', random_state=0, **bp).fit(X[trv], y[trv])
m = te & base.notna()
print('baseline test:', mae(y[m], base[m]), '| model final test:', mae(y[te], final.predict(X[te])))

In [ ]:
pt = pd.Series(final.predict(X[te]), index=X.index[te]); w = slice('2026-06-15', '2026-06-21')
y[te][w].plot(label='aktual'); pt[w].plot(label='prediksi'); plt.legend(); plt.ylabel('kWh');

Penyimpanan model dan metadata dikerjakan oleh `python src/train.py` (logika yang sama dengan sel di atas).